# MaS-TransUNet — Colab Training Notebook

This notebook trains and evaluates the MaS-TransUNet model for medical image segmentation.
Run each cell in order. Read the markdown instructions before each cell carefully.

---
## 1. Mount Google Drive

This mounts your Google Drive to `/content/drive` so datasets and checkpoints persist across sessions.

**Before proceeding:** Make sure you are logged into the Google account that contains your datasets under `My Drive/datasets/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 2. Install Dependencies

Installs all required packages and verifies CUDA is available.

**Before proceeding:** Check that the output shows a GPU name (e.g. `Tesla T4`). If it says CUDA is not available, go to `Runtime > Change runtime type` and select **GPU**.

In [ ]:
!pip install -q timm==0.9.12 einops albumentations \
    opencv-python-headless scikit-image scikit-learn monai matplotlib \
    seaborn pandas tqdm tensorboard scipy Pillow

import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is NOT available. Training will be very slow.")
    print("Go to Runtime > Change runtime type > GPU")

---
## 3. Clone Repository

Clones the MaS-TransUNet repository from GitHub and enters the project directory.

**Repository URL:** `https://github.com/Surfing-Ninja/TransUNet.git`
On reconnection, `git pull` ensures you have the latest code.

In [ ]:
import os

REPO_URL = "https://github.com/Surfing-Ninja/TransUNet.git"
REPO_DIR = "/content/TransUNet"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
!git pull
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Auto-unzip and organize datasets from Google Drive
# Run this once after mounting Drive (Cell 1).
# Behavior:
# - Scans ONLY MyDrive/datasets for .zip files
# - Skips unchanged zips that were already processed
# - Normalizes datasets to train/test images+masks when needed

import os
import re
import json
import zipfile
import shutil
import random
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive"
DATASETS_ROOT = os.path.join(DRIVE_ROOT, "datasets")
RAW_ROOT = os.path.join(DATASETS_ROOT, "_raw_extracted")
UNASSIGNED_ROOT = os.path.join(DATASETS_ROOT, "_unassigned")
STATE_PATH = os.path.join(DATASETS_ROOT, "_zip_processing_state.json")
FORCE_REPROCESS = False  # set True only when you want to rebuild from scratch

os.makedirs(DATASETS_ROOT, exist_ok=True)
os.makedirs(RAW_ROOT, exist_ok=True)
os.makedirs(UNASSIGNED_ROOT, exist_ok=True)

EXPECTED = ["tcga_lgg", "dsb2018", "kvasir_seg", "isic2018", "covid_ct"]

# Optional manual override when zip names are generic
# Example:
# MANUAL_MAP = {
#   "archive (4).zip": "covid_ct",
#   "archive (5).zip": "tcga_lgg",
# }
MANUAL_MAP = {}

KEYWORDS = {
    "tcga_lgg": ["tcga", "lgg", "glioma"],
    "dsb2018": ["dsb", "bowl", "nuclei", "2018"],
    "kvasir_seg": ["kvasir", "polyp"],
    "isic2018": ["isic", "lesion", "skin"],
    "covid_ct": ["covid", "lung", "infection"],
}


def list_zip_files_in_datasets(root_dir):
    zips = []
    for p in Path(root_dir).rglob("*.zip"):
        parts = set(p.parts)
        if "_raw_extracted" in parts or "_unassigned" in parts:
            continue
        zips.append(str(p))
    return sorted(set(zips))


def zip_signature(zip_path):
    st = os.stat(zip_path)
    return f"{st.st_size}:{int(st.st_mtime)}"


def load_state(path):
    if not os.path.isfile(path):
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            return data if isinstance(data, dict) else {}
    except Exception:
        return {}


def save_state(path, state):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2, sort_keys=True)


def score_dataset(text: str, dataset_name: str) -> int:
    return sum(1 for k in KEYWORDS[dataset_name] if k in text)


def infer_dataset_name(zip_path, lower_names, used_names):
    base = os.path.basename(zip_path)

    if base in MANUAL_MAP:
        mapped = MANUAL_MAP[base]
        if mapped in EXPECTED and mapped not in used_names:
            return mapped

    text = f"{base.lower()} {' '.join(lower_names[:1000])}"
    scored = [(ds, score_dataset(text, ds)) for ds in EXPECTED if ds not in used_names]
    scored.sort(key=lambda x: x[1], reverse=True)

    if scored and scored[0][1] > 0:
        return scored[0][0]
    return None


def pick_candidate_root(extract_dir):
    best = extract_dir
    for root, dirs, _ in os.walk(extract_dir):
        d = {x.lower() for x in dirs}
        if {"train", "test"}.issubset(d):
            return root
        if any(k in d for k in [
            "images", "image", "imgs", "masks", "mask", "labels", "label",
            "ground_truth", "gt", "ct_scans", "infection_mask", "lung_mask", "lung_and_infection_mask"
        ]):
            best = root

    items = os.listdir(extract_dir)
    if len(items) == 1:
        only = os.path.join(extract_dir, items[0])
        if os.path.isdir(only):
            return only

    return best


def find_first_dir_with_any(root_dir, candidates):
    for c in candidates:
        p = os.path.join(root_dir, c)
        if os.path.isdir(p):
            return p

    for cur, dirs, _ in os.walk(root_dir):
        low = {d.lower() for d in dirs}
        for c in candidates:
            if c.lower() in low:
                for d in dirs:
                    if d.lower() == c.lower():
                        return os.path.join(cur, d)
    return None


def list_image_files(folder):
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".nii", ".nii.gz", ".gz")
    out = []
    for p in Path(folder).rglob("*"):
        if p.is_file() and p.name.lower().endswith(exts):
            out.append(str(p))
    return sorted(out)


def stem_key(path):
    base = os.path.basename(path).lower()
    for ext in [".nii.gz", ".nii", ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".gz"]:
        if base.endswith(ext):
            base = base[:-len(ext)]
            break
    return re.sub(r"(_mask|_masks|_seg|_label|_labels|mask|seg|_infection|infection|_lung|lung)$", "", base)


def match_pairs(image_files, mask_files):
    mask_map = {stem_key(m): m for m in mask_files}
    pairs = []
    for img in image_files:
        key = stem_key(img)
        if key in mask_map:
            pairs.append((img, mask_map[key]))
    return pairs


def copy_split_pairs(pairs, train_images, train_masks, test_images, test_masks, seed=42, test_ratio=0.1):
    random.seed(seed)
    random.shuffle(pairs)

    n_total = len(pairs)
    n_test = max(1, int(n_total * test_ratio)) if n_total > 1 else 0
    n_train = n_total - n_test

    train_pairs = pairs[:n_train]
    test_pairs = pairs[n_train:]

    for folder in [train_images, train_masks, test_images, test_masks]:
        os.makedirs(folder, exist_ok=True)

    def get_ext(p):
        n = os.path.basename(p).lower()
        if n.endswith(".nii.gz"): return ".nii.gz"
        return os.path.splitext(n)[1] or ".png"

    def copy_pairs(pairs_list, img_out, mask_out):
        for i, (img, msk) in enumerate(pairs_list):
            name = f"{i:05d}"
            shutil.copy2(img, os.path.join(img_out, name + get_ext(img)))
            shutil.copy2(msk, os.path.join(mask_out, name + get_ext(msk)))

    copy_pairs(train_pairs, train_images, train_masks)
    copy_pairs(test_pairs, test_images, test_masks)

    return len(train_pairs), len(test_pairs)


def dataset_ready(ds_dir):
    required = [
        os.path.join(ds_dir, "train", "images"),
        os.path.join(ds_dir, "train", "masks"),
        os.path.join(ds_dir, "test", "images"),
        os.path.join(ds_dir, "test", "masks"),
    ]
    return all(os.path.isdir(p) for p in required)


def ensure_structure(dataset_dir, seed=42, test_ratio=0.1):
    train_images = os.path.join(dataset_dir, "train", "images")
    train_masks = os.path.join(dataset_dir, "train", "masks")
    test_images = os.path.join(dataset_dir, "test", "images")
    test_masks = os.path.join(dataset_dir, "test", "masks")

    if all(os.path.isdir(p) for p in [train_images, train_masks, test_images, test_masks]):
        os.makedirs(os.path.join(dataset_dir, "train", "edges"), exist_ok=True)
        os.makedirs(os.path.join(dataset_dir, "test", "edges"), exist_ok=True)
        return "already-structured"

    # val/valid -> test conversion
    val_images = os.path.join(dataset_dir, "val", "images")
    val_masks = os.path.join(dataset_dir, "val", "masks")
    valid_images = os.path.join(dataset_dir, "valid", "images")
    valid_masks = os.path.join(dataset_dir, "valid", "masks")

    if os.path.isdir(train_images) and os.path.isdir(train_masks):
        if os.path.isdir(val_images) and os.path.isdir(val_masks):
            os.makedirs(os.path.join(dataset_dir, "test"), exist_ok=True)
            if os.path.exists(test_images):
                shutil.rmtree(test_images)
            if os.path.exists(test_masks):
                shutil.rmtree(test_masks)
            shutil.move(val_images, test_images)
            shutil.move(val_masks, test_masks)
            shutil.rmtree(os.path.join(dataset_dir, "val"), ignore_errors=True)
        elif os.path.isdir(valid_images) and os.path.isdir(valid_masks):
            os.makedirs(os.path.join(dataset_dir, "test"), exist_ok=True)
            if os.path.exists(test_images):
                shutil.rmtree(test_images)
            if os.path.exists(test_masks):
                shutil.rmtree(test_masks)
            shutil.move(valid_images, test_images)
            shutil.move(valid_masks, test_masks)
            shutil.rmtree(os.path.join(dataset_dir, "valid"), ignore_errors=True)

    if all(os.path.isdir(p) for p in [train_images, train_masks, test_images, test_masks]):
        os.makedirs(os.path.join(dataset_dir, "train", "edges"), exist_ok=True)
        os.makedirs(os.path.join(dataset_dir, "test", "edges"), exist_ok=True)
        return "converted-val-to-test"

    # Special handling for COVID-CT style structure
    ct_dir = find_first_dir_with_any(dataset_dir, ["ct_scans", "ct_scan", "ct", "scans"])
    covid_mask_dir = None
    for candidate in ["lung_and_infection_mask", "infection_mask", "lung_mask"]:
        covid_mask_dir = find_first_dir_with_any(dataset_dir, [candidate])
        if covid_mask_dir:
            break

    if ct_dir and covid_mask_dir:
        img_files = list_image_files(ct_dir)
        mask_files = list_image_files(covid_mask_dir)
        pairs = match_pairs(img_files, mask_files)

        if not pairs:
            raise RuntimeError(f"COVID layout detected but no matching scan/mask pairs in {dataset_dir}")

        n_train, n_test = copy_split_pairs(
            pairs, train_images, train_masks, test_images, test_masks,
            seed=seed, test_ratio=test_ratio,
        )

        os.makedirs(os.path.join(dataset_dir, "train", "edges"), exist_ok=True)
        os.makedirs(os.path.join(dataset_dir, "test", "edges"), exist_ok=True)
        return f"converted-covid-layout ({n_train} train / {n_test} test)"

    # Generic fallback: images + masks in arbitrary depth
    images_dir = find_first_dir_with_any(dataset_dir, ["images", "image", "imgs"])
    masks_dir = find_first_dir_with_any(dataset_dir, ["masks", "mask", "labels", "label", "ground_truth", "gt"])

    if not images_dir or not masks_dir:
        raise RuntimeError(f"Could not find image/mask folders in {dataset_dir}")

    img_files = list_image_files(images_dir)
    mask_files = list_image_files(masks_dir)
    pairs = match_pairs(img_files, mask_files)

    if not pairs:
        raise RuntimeError(f"No matching image/mask pairs found in {dataset_dir}")

    n_train, n_test = copy_split_pairs(
        pairs, train_images, train_masks, test_images, test_masks,
        seed=seed, test_ratio=test_ratio,
    )

    os.makedirs(os.path.join(dataset_dir, "train", "edges"), exist_ok=True)
    os.makedirs(os.path.join(dataset_dir, "test", "edges"), exist_ok=True)

    return f"split-flat-layout ({n_train} train / {n_test} test)"


def count_images(folder):
    if not os.path.isdir(folder):
        return 0
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".nii", ".nii.gz", ".gz")
    return sum(1 for p in Path(folder).rglob("*") if p.is_file() and p.name.lower().endswith(exts))


zip_paths = list_zip_files_in_datasets(DATASETS_ROOT)
if not zip_paths:
    print("No zip files found — assuming datasets are already extracted and structured.")
    print("Skipping Cell 8. Proceed to Cell 4 to verify dataset paths.")

print("Found zip files under MyDrive/datasets:")
for p in zip_paths:
    print(" -", p)

state = load_state(STATE_PATH)
summary = []
used = set()

for zp in zip_paths:
    base_zip = os.path.basename(zp)
    sig = zip_signature(zp)
    state_entry = state.get(zp, {})

    # Skip unchanged archives that already produced a ready dataset
    if not FORCE_REPROCESS and state_entry.get("signature") == sig and state_entry.get("status") == "ok":
        mapped_ds = state_entry.get("dataset")
        if mapped_ds and dataset_ready(os.path.join(DATASETS_ROOT, mapped_ds)):
            summary.append((base_zip, mapped_ds, "SKIPPED (already processed, unchanged zip)"))
            print(f"[SKIP] {base_zip}: unchanged and already prepared")
            used.add(mapped_ds)
            continue

    safe_name = re.sub(r"[^a-zA-Z0-9._-]+", "_", os.path.splitext(base_zip)[0])
    raw_extract_dir = os.path.join(RAW_ROOT, f"{safe_name}__{sig.replace(':', '_')}")
    if os.path.isdir(raw_extract_dir):
        shutil.rmtree(raw_extract_dir)
    os.makedirs(raw_extract_dir, exist_ok=True)

    print(f"\n[UNZIP] {base_zip} -> {raw_extract_dir}")
    with zipfile.ZipFile(zp, "r") as zf:
        names = [n.lower() for n in zf.namelist() if n and not n.endswith("/")]
        zf.extractall(raw_extract_dir)

    ds_name = infer_dataset_name(zp, names, used)
    if ds_name is None:
        unassigned_dir = os.path.join(UNASSIGNED_ROOT, safe_name)
        if os.path.isdir(unassigned_dir):
            shutil.rmtree(unassigned_dir)
        shutil.move(raw_extract_dir, unassigned_dir)

        state[zp] = {
            "signature": sig,
            "status": "unassigned",
            "dataset": None,
            "note": f"saved to {unassigned_dir}",
        }
        summary.append((base_zip, "UNASSIGNED", f"saved to {unassigned_dir}"))
        print(f"[SKIP] Could not infer dataset name. Extracted under: {unassigned_dir}")
        continue

    if ds_name in used:
        unassigned_dir = os.path.join(UNASSIGNED_ROOT, safe_name)
        if os.path.isdir(unassigned_dir):
            shutil.rmtree(unassigned_dir)
        shutil.move(raw_extract_dir, unassigned_dir)

        state[zp] = {
            "signature": sig,
            "status": "duplicate",
            "dataset": ds_name,
            "note": f"duplicate mapping; moved to {unassigned_dir}",
        }
        summary.append((base_zip, ds_name, "SKIPPED duplicate mapping (moved to _unassigned)"))
        print(f"[SKIP] Duplicate mapping to {ds_name}. Moved to: {unassigned_dir}")
        continue

    used.add(ds_name)
    ds_dir = os.path.join(DATASETS_ROOT, ds_name)
    candidate_root = pick_candidate_root(raw_extract_dir)

    if os.path.isdir(ds_dir):
        shutil.rmtree(ds_dir)
    os.makedirs(ds_dir, exist_ok=True)

    for item in os.listdir(candidate_root):
        src = os.path.join(candidate_root, item)
        dst = os.path.join(ds_dir, item)
        if os.path.isdir(src):
            shutil.move(src, dst)
        else:
            if not os.path.exists(dst):
                shutil.move(src, dst)

    try:
        result = ensure_structure(ds_dir)
        n_train = count_images(os.path.join(ds_dir, "train", "images"))
        n_test = count_images(os.path.join(ds_dir, "test", "images"))

        state[zp] = {
            "signature": sig,
            "status": "ok",
            "dataset": ds_name,
            "result": result,
            "train_count": n_train,
            "test_count": n_test,
        }
        summary.append((base_zip, ds_name, result, n_train, n_test))
        print(f"[OK] {base_zip} -> {ds_name} | {result} | train={n_train}, test={n_test}")
    except Exception as exc:
        state[zp] = {
            "signature": sig,
            "status": "failed",
            "dataset": ds_name,
            "error": str(exc),
        }
        summary.append((base_zip, ds_name, f"FAILED: {exc}"))
        print(f"[WARN] {base_zip} -> {ds_name} failed: {exc}")

save_state(STATE_PATH, state)

print("\n=== Dataset Preparation Summary ===")
for row in summary:
    print("-", row)

print("\nZip processing state saved to:", STATE_PATH)
print("All archives are extracted under MyDrive/datasets (or _unassigned).")
print("Next: run Cell 4 and confirm datasets show Found.")

---
## 4. Set Colab Mode

Switches all config paths to Google Drive locations and sets the device to CUDA.

**Before proceeding:** Verify each dataset path prints **Found**. If any path prints **Missing**, upload the dataset to the corresponding Drive folder before training on that dataset.

In [ ]:
import os
import importlib
import config
importlib.reload(config)
from config import CFG

# Switch to Colab paths
CFG.is_colab = True
CFG.__post_init__()  # Re-derive paths with Colab flag
CFG.device = "cuda"

print(f"Base data dir:   {CFG.base_data_dir}")
print(f"Checkpoint dir:  {CFG.checkpoint_dir}")
print(f"Device:          {CFG.device}")
print()

for name, paths in CFG.dataset_paths.items():
    img_dir = paths['train_images']
    status = "Found" if os.path.isdir(img_dir) else "Missing"
    print(f"  {name:12s}  {status:7s}  {img_dir}")

---
## 5. Create Directories

Creates the checkpoint and log directories on Drive if they do not already exist.

**Before proceeding:** Confirm both directories show `exists: True`.

In [ ]:
import os
from config import CFG

os.makedirs(CFG.checkpoint_dir, exist_ok=True)
os.makedirs(CFG.log_dir, exist_ok=True)

print(f"Checkpoint dir: {CFG.checkpoint_dir}  (exists: {os.path.isdir(CFG.checkpoint_dir)})")
print(f"Log dir:        {CFG.log_dir}  (exists: {os.path.isdir(CFG.log_dir)})")

---
## 6. Preprocess Datasets

Generates Canny edge maps for all training and test masks. This only runs if edge map directories are empty or missing, so it is safe to re-run after a Colab reconnection without redoing work.

**Before proceeding:** Check the summary. Each dataset with data present should show a non-zero edge map count. Datasets marked as not found are skipped.

In [ ]:
import os
from config import CFG

needs_preprocessing = False
for name, paths in CFG.dataset_paths.items():
    edge_dir = paths['train_edges']
    masks_dir = paths['train_masks']
    if os.path.isdir(masks_dir):
        if not os.path.isdir(edge_dir) or len(os.listdir(edge_dir)) == 0:
            needs_preprocessing = True
            print(f"  {name}: edge maps missing -- will preprocess")
        else:
            print(f"  {name}: edge maps already exist ({len(os.listdir(edge_dir))} files) -- skipping")
    else:
        print(f"  {name}: dataset not found -- skipping")

if needs_preprocessing:
    print("\nRunning preprocessing...")
    !python preprocess.py all
else:
    print("\nAll edge maps already exist. No preprocessing needed.")

---
## 7. Smoke Test

Instantiates the MaS-TransUNet model, runs a single forward pass with dummy data, and prints output shapes and parameter count. This catches import errors, shape mismatches, or GPU memory issues before committing to a full training run.

**Before proceeding:** Verify all four output shapes are printed, the parameter count is ~117M, and there are no errors.

In [ ]:
import torch
from config import CFG
from models import build_model

model = build_model(CFG)

dummy_image = torch.randn(1, 3, 224, 224, device=CFG.device)
dummy_mask  = torch.zeros(1, 1, 224, 224, device=CFG.device)

with torch.no_grad():
    outputs = model(dummy_image, dummy_mask)

print("Output shapes:")
for key, val in outputs.items():
    print(f"  {key:10s}: {tuple(val.shape)}")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params / 1e6:.2f}M")
print("Smoke test PASSED")

# Free memory
del model, dummy_image, dummy_mask, outputs
torch.cuda.empty_cache()

---
## 8. Auto-Select Datasets and Train

Automatically detects all datasets that are present and valid, then trains MaS-TransUNet on each one sequentially.

Training automatically resumes from the latest checkpoint if a Colab session disconnects. Checkpoints and previous-epoch masks are saved to Google Drive every 5 epochs.

**Before proceeding:** Ensure Cell 4 shows **Found** for at least one dataset.

In [ ]:
import os
from config import CFG

# Set to None to train all found datasets, or a specific name to train just one
TRAIN_ONLY = None

# Auto-detect datasets that actually exist and train them one by one
available_datasets = []
for name, paths in CFG.dataset_paths.items():
    train_images = paths["train_images"]
    train_masks = paths["train_masks"]
    test_images = paths["test_images"]
    test_masks = paths["test_masks"]

    if all(os.path.isdir(p) for p in [train_images, train_masks, test_images, test_masks]):
        available_datasets.append(name)

if TRAIN_ONLY:
    available_datasets = [d for d in available_datasets if d == TRAIN_ONLY]

print("Datasets found:", available_datasets)

if not available_datasets:
    raise RuntimeError(
        "No valid datasets found. Run the dataset unzip/organize cell and ensure Cell 4 shows Found."
    )

for DATASET_NAME in available_datasets:
    print(f"\n=== Training: {DATASET_NAME} ===")
    !python train.py {DATASET_NAME}

print("\nAll available datasets have finished training.")

### 8.1 Verify Saved Checkpoints

Validates that checkpoints were actually written for each trained dataset and that files can be loaded.

In [ ]:
import os
import glob
import re
import torch
import numpy as np
from config import CFG

# Prefer dataset list from training cell; fallback to filenames in checkpoint dir
if "available_datasets" in globals() and len(available_datasets) > 0:
    datasets_to_check = list(available_datasets)
else:
    pattern = os.path.join(CFG.checkpoint_dir, "*_best.pth")
    datasets_to_check = sorted({os.path.basename(p).replace("_best.pth", "") for p in glob.glob(pattern)})

if not datasets_to_check:
    raise RuntimeError(
        f"No trained datasets found in {CFG.checkpoint_dir}. Run Cell 8 first."
    )

print(f"Checkpoint directory: {CFG.checkpoint_dir}")
print("Datasets to verify:", datasets_to_check)

required_keys = {
    "model_state_dict",
    "optimizer_state_dict",
    "scheduler_state_dict",
    "epoch",
    "best_dice",
    "dataset_name",
}

all_ok = True
for ds in datasets_to_check:
    print(f"\n=== Verifying {ds} ===")

    periodic_ckpts = sorted(glob.glob(os.path.join(CFG.checkpoint_dir, f"{ds}_[0-9][0-9][0-9].pth")))
    best_ckpt = os.path.join(CFG.checkpoint_dir, f"{ds}_best.pth")
    prev_masks = os.path.join(CFG.checkpoint_dir, f"{ds}_prev_masks.npz")

    if not periodic_ckpts:
        print("[WARN] No periodic epoch checkpoints found")
        all_ok = False
    else:
        latest = periodic_ckpts[-1]
        ckpt = torch.load(latest, map_location="cpu", weights_only=False)
        missing = required_keys - set(ckpt.keys())
        if missing:
            print(f"[FAIL] Latest epoch checkpoint missing keys: {sorted(missing)}")
            all_ok = False
        else:
            print(f"[OK] Latest epoch checkpoint readable: {os.path.basename(latest)}")
            print(f"     epoch={ckpt['epoch']}, best_dice={ckpt['best_dice']:.6f}")

    if not os.path.isfile(best_ckpt):
        print("[WARN] Best checkpoint file missing")
        all_ok = False
    else:
        best = torch.load(best_ckpt, map_location="cpu", weights_only=False)
        missing = required_keys - set(best.keys())
        if missing:
            print(f"[FAIL] Best checkpoint missing keys: {sorted(missing)}")
            all_ok = False
        else:
            print(f"[OK] Best checkpoint readable: {os.path.basename(best_ckpt)}")
            print(f"     epoch={best['epoch']}, best_dice={best['best_dice']:.6f}")

    if not os.path.isfile(prev_masks):
        print("[WARN] Previous-epoch masks file missing")
        all_ok = False
    else:
        data = np.load(prev_masks)
        print(f"[OK] Prev-mask cache readable: {os.path.basename(prev_masks)} (entries={len(data.files)})")

if all_ok:
    print("\nAll checkpoints verified successfully.")
else:
    print("\nVerification completed with warnings/failures. Check logs above.")

---
## 9. Evaluate

Runs evaluation on the test set using the best checkpoint and iterative FAM refinement (up to 10 iterations per sample). Prints a metrics table matching Tables II–VI in the paper, saves per-sample results to CSV, and writes qualitative side-by-side PNGs for the first 5 test images.

**Before proceeding:** Make sure training (Cell 8) has completed at least a few epochs and a `_best.pth` checkpoint exists.

In [ ]:
if "available_datasets" not in globals() or not available_datasets:
    print("No datasets available to evaluate. Make sure to run the training cell first.")
else:
    for dataset in available_datasets:
        print(f"\n=== Evaluating: {dataset} ===")
        !python evaluate.py {dataset}

---
## 10. Training Order Note

The following cell explains the recommended order for training across all five datasets.

### Recommended Training Order

Train datasets in this order for fastest iteration and debugging:

| Order | Dataset      | Size     | Notes |
|:-----:|:-------------|:---------|:------|
| 1     | `tcga_lgg`   | Smallest | Start here to verify the full pipeline end-to-end with minimal time. Good for catching bugs early. |
| 2     | `dsb2018`    | Small    | Data Science Bowl 2018 nuclei segmentation. Quick to train, validates generalisation to cell-level tasks. |
| 3     | `kvasir_seg` | Medium   | Gastrointestinal polyp segmentation with more complex boundaries. Tests the EAM and edge loss. |
| 4     | `isic2018`   | Large    | Skin lesion segmentation with high variability. Longer training; monitor for overfitting. |
| 5     | `covid_ct`   | Largest  | COVID-19 CT scan segmentation. Most computationally expensive. Train last once everything else works. |

> To train a single dataset: Go to Cell 18, set `TRAIN_ONLY = "tcga_lgg"` (or whichever dataset you want), then re-run Cell 18.

**Checkpointing:** All checkpoints are saved to Google Drive, so training resumes automatically after a Colab disconnection. Simply re-run from Cell 1.